# Chapter 6: Adding memory to your agent

In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## 6.1 The anatomy of agent memory

## 6.2 Managing context during execution

### 6.2.1 Separating storage from presentation

**Listing 6.1** Creating an optimizer callback with a threshold

In [ ]:
def create_optimizer_callback(apply_optimization, threshold: int = 50000):
    """Factory function that creates a callback applying optimization strategy"""
    async def callback(
        context: ExecutionContext,
        request: LlmRequest
    ) -> Optional[LlmResponse]:
        token_count = count_tokens(request)
        if token_count < threshold:
            return None
        # Support both sync and async functions
        result = apply_optimization(context, request)
        if inspect.isawaitable(result):
            await result
        return None
    return callback

### 6.2.2 Sliding window strategy

**Listing 6.2** Sliding window strategy implementation

In [ ]:
def apply_sliding_window(
    context: ExecutionContext,
    request: LlmRequest,
    window_size: int = 20
) -> None:
    """Sliding window that keeps only the most recent N messages"""
    
    contents = request.contents
    
    # Find user message position
    user_message_idx = None
    for i, item in enumerate(contents):
        if isinstance(item, Message) and item.role == "user":
            user_message_idx = i
            break
    
    if user_message_idx is None:
        return
    
    # Preserve up to user message
    preserved = contents[:user_message_idx + 1]
    
    # Keep only the most recent N from remaining items
    remaining = contents[user_message_idx + 1:]
    if len(remaining) > window_size:
        remaining = remaining[-window_size:]
    
    request.contents = preserved + remaining

**Listing 6.3** Using a sliding window with optimizer callback

In [ ]:
optimizer = create_optimizer_callback(
    apply_optimization=apply_sliding_window,
    threshold=30000
)
agent = Agent(
    model=LlmClient(model="gpt-5"),
    before_llm_callback=optimizer
)

### 6.2.3 Token counting

**Listing 6.4** Token counting with tiktoken

In [ ]:
def count_tokens(request: LlmRequest) -> int:
    """Calculate total token count of LlmRequest"""
    import tiktoken
    from .llm import build_messages
    # Select encoding for model, use default on failure
    try:
        encoding = tiktoken.encoding_for_model(request.model_id or "gpt-5")
    except KeyError:
        encoding = tiktoken.get_encoding("o200k_base")
    # Convert to API message format then count tokens
    messages = build_messages(request)
    total_tokens = 0
    for message in messages:
        # Per-message overhead (role, separators, etc.)
        total_tokens += 4
        # Content tokens
        if message.get("content"):
            total_tokens += len(encoding.encode(message["content"]))
        # tool_calls tokens
        if message.get("tool_calls"):
            for tool_call in message["tool_calls"]:
                func = tool_call.get("function", {})
                if func.get("name"):
                    total_tokens += len(encoding.encode(func["name"]))
                if func.get("arguments"):
                    total_tokens += len(encoding.encode(func["arguments"]))
    # Tool definition tokens
    if request.tools:
        for tool in request.tools:
            tool_def = tool.tool_definition
            total_tokens += len(encoding.encode(json.dumps(tool_def)))
    return total_tokens

### 6.2.4 Compaction strategy

**Listing 6.5** Compaction strategy implementation

In [ ]:
# Tools to compress ToolCall arguments
TOOLCALL_COMPACTION_RULES = {
    "create_file": "[Content saved to file]",
}
# Tools to compress ToolResult content
TOOLRESULT_COMPACTION_RULES = {
    "read_file": "File content from {file_path}. Re-read if needed.",
    "search_web": "Search results processed. Query: {query}. Re-search if needed.",
    "tavily_search": "Search results processed. Query: {query}. Re-search if needed.",
}
def apply_compaction(context: ExecutionContext, request: LlmRequest) -> None:
    """Compress tool calls and results into reference messages"""
    
    tool_call_args: Dict[str, Dict] = {}
    compacted = []
    
    for item in request.contents:
        if isinstance(item, ToolCall):
            # Save arguments (for use when compressing ToolResult later)
            tool_call_args[item.tool_call_id] = item.arguments
            
            # If the ToolCall itself is a compression target (create_file, etc.)
            if item.name in TOOLCALL_COMPACTION_RULES:
                compressed_args = {
                    k: TOOLCALL_COMPACTION_RULES[item.name] if k == "content" else v
                    for k, v in item.arguments.items()
                }
                compacted.append(ToolCall(
                    tool_call_id=item.tool_call_id,
                    name=item.name,
                    arguments=compressed_args
                ))
            else:
                compacted.append(item)
                
        elif isinstance(item, ToolResult):
            # If ToolResult is a compression target (read_file, search_web, etc.)
            if item.name in TOOLRESULT_COMPACTION_RULES:
                args = tool_call_args.get(item.tool_call_id, {})
                template = TOOLRESULT_COMPACTION_RULES[item.name]
                compressed_content = template.format(
                    file_path=args.get("file_path", args.get("path", "unknown")),
                    query=args.get("query", "unknown")
                )
                compacted.append(ToolResult(
                    tool_call_id=item.tool_call_id,
                    name=item.name,
                    status=item.status,
                    content=[compressed_content]
                ))
            else:
                compacted.append(item)
        else:
            compacted.append(item)
    
    request.contents = compacted

### 6.2.5 Summarization strategy

**Listing 6.6** Summarization strategy implementation

In [ ]:
async def apply_summarization(
    context: ExecutionContext,
    request: LlmRequest,
    llm_client: LlmClient,
    keep_recent: int = 5
) -> None:
    """Replace old messages with a summary"""
    contents = request.contents
    # Find user message position
    user_idx = None
    for i, item in enumerate(contents):
        if isinstance(item, Message) and item.role == "user":
            user_idx = i
            break
    if user_idx is None:
        return
    # Check previous summary position (skip already-summarized portions)
    last_summary_idx = context.state.get("last_summary_idx", user_idx)
    # Calculate summarization target range
    summary_start = last_summary_idx + 1
    summary_end = len(contents) - keep_recent
    # Overlap prevention: exit if nothing to summarize or range is invalid
    if summary_end <= summary_start:
        return
    # Determine portions to preserve (no overlap)
    preserved_start = contents[:last_summary_idx + 1]
    preserved_end = contents[summary_end:]
    to_summarize = contents[summary_start:summary_end]
    # Generate summary
    history_text = format_history_for_summary(to_summarize)
    summary = await generate_summary(llm_client, history_text)
    # Add summary to instructions
    request.append_instructions(f"[Previous work summary]\n{summary}")
    # Keep only preserved portions in contents
    request.contents = preserved_start + preserved_end
    # Record summary position
    context.state["last_summary_idx"] = len(preserved_start) - 1
def format_history_for_summary(items: List[ContentItem]) -> str:
    """Convert ContentItem list to text for summarization"""
    lines = []
    for item in items:
        if isinstance(item, Message):
            lines.append(f"[{item.role}]: {item.content[:500]}...")
        elif isinstance(item, ToolCall):
            lines.append(f"[Tool Call]: {item.name}({item.arguments})")
        elif isinstance(item, ToolResult):
            content_preview = str(item.content[0])[:200] if item.content else ""
            lines.append(f"[Tool Result]: {item.name} -> {content_preview}...")
    return "\n".join(lines)
async def generate_summary(llm_client: LlmClient, history: str) -> str:
    """Generate history summary using LLM"""
    
    request = LlmRequest(
        instructions=[SUMMARIZATION_PROMPT.format(history=history)],
        contents=[Message(role="user", content="Please summarize.")]
    )
    
    response = await llm_client.generate(request)
    
    for item in response.content:
        if isinstance(item, Message):
            return item.content
    
    return ""

### 6.2.6 Hierarchical context management

**Listing 6.7** Hierarchical ContextOptimizer class

In [ ]:
class ContextOptimizer:
    """Hierarchical context optimization strategy"""
    def __init__(
        self,
        llm_client: LlmClient,
        token_threshold: int = 50000,
        enable_compaction: bool = True,
        enable_summarization: bool = True,
        keep_recent: int = 5
    ):
        self.llm_client = llm_client
        self.token_threshold = token_threshold
        self.enable_compaction = enable_compaction
        self.enable_summarization = enable_summarization
        self.keep_recent = keep_recent
    async def __call__(
        self,
        context: ExecutionContext,
        request: LlmRequest
    ) -> Optional[LlmResponse]:
        """Register as before_llm_callback"""
        # Step 1: Measure tokens
        if count_tokens(request) < self.token_threshold:
            return None
        # Step 2: Apply Compaction
        if self.enable_compaction:
            apply_compaction(context, request)
            if count_tokens(request) < self.token_threshold:
                return None
        # Step 3: Apply Summarization
        if self.enable_summarization:
            await apply_summarization(
                context,
                request,
                self.llm_client,
                self.keep_recent
            )
        return None

## 6.3 Continuous execution: Session and state management

### 6.3.1 The session class

**Listing 6.8** Session class design

In [ ]:
class Session(BaseModel):
    """Container for persistent conversation state across multiple run() calls."""
    
    session_id: str
    user_id: str | None = None
    events: list[Event] = Field(default_factory=list)
    state: dict[str, Any] = Field(default_factory=dict)
    created_at: datetime = Field(default_factory=datetime.now)
    updated_at: datetime = Field(default_factory=datetime.now)

### 6.3.2 Managing sessions with SessionManager

**Listing 6.9** BaseSessionManager abstract class

In [ ]:
from abc import ABC, abstractmethod
class BaseSessionManager(ABC):
    """Abstract base class for session management."""
    
    @abstractmethod
    async def create(
        self, 
        session_id: str, 
        user_id: str | None = None
    ) -> Session:
        """Create a new session."""
        pass
    
    @abstractmethod
    async def get(self, session_id: str) -> Session | None:
        """Retrieve a session by ID. Returns None if not found."""
        pass
    
    @abstractmethod
    async def save(self, session: Session) -> None:
        """Persist session changes to storage."""
        pass
    
    async def get_or_create(
        self, 
        session_id: str, 
        user_id: str | None = None
    ) -> Session:
        """Get existing session or create new one."""
        session = await self.get(session_id)
        if session is None:
            session = await self.create(session_id, user_id)
        return session

**Listing 6.10** InMemorySessionManager implementation

In [ ]:
class InMemorySessionManager(BaseSessionManager):
    """In-memory session storage for development and testing."""
    
    def __init__(self):
        self._sessions: dict[str, Session] = {}
    
    async def create(
        self, 
        session_id: str, 
        user_id: str | None = None
    ) -> Session:
        """Create a new session."""
        if session_id in self._sessions:
            raise ValueError(f"Session {session_id} already exists")
        
        session = Session(
            session_id=session_id,
            user_id=user_id
        )
        self._sessions[session_id] = session
        return session
    
    async def get(self, session_id: str) -> Session | None:
        """Retrieve a session by ID."""
        return self._sessions.get(session_id)
    
    async def save(self, session: Session) -> None:
        """Save session to storage."""
        self._sessions[session.session_id] = session

### 6.3.3 Integrating sessions into the agent

**Listing 6.11** ExecutionContext with session support

In [ ]:
@dataclass
class ExecutionContext:
    # Existing attributes...
    
    # Session support
    session: Session | None = None
    session_manager: BaseSessionManager | None = None
    
    @property
    def events(self) -> list[Event]:
        if self.session:
            return self.session.events
    
    @property
    def state(self) -> dict[str, Any]:
        if self.session:
            return self.session.state
    
    def add_event(self, event: Event):
        """Append an event to the execution history."""
        self.events.append(event)
        # Update session timestamp if available
        if self.session:
            self.session.updated_at = datetime.now()

**Listing 6.12** Agent initialization with SessionManager

In [ ]:
class ToolCallingAgent:
    """Tool-calling agent with session support."""
    
    def __init__(
        self,
        model: LlmClient,
        tools: list[BaseTool] | None = None,
        instructions: str = "",
        max_steps: int = 10,
        session_manager: BaseSessionManager | None = None,
    ):
        self.model = model
        self.instructions = instructions
        self.max_steps = max_steps
        self.tools = self._setup_tools(tools or [])
        
        # Use provided session manager or create default
        self.session_manager = session_manager or InMemorySessionManager()

**Listing 6.13** run() method with session support

In [ ]:
async def run(
    self, 
    user_input: str,
    session_id: str | None = None,
    context: ExecutionContext | None = None,
) -> AgentResult:
    """Execute the agent with optional session support."""
    
    # Load or create session if session_id is provided
    session = None
    if session_id:
        session = await self.session_manager.get_or_create(session_id)
    
    # Create context with session support
    if context is None:
        context = ExecutionContext(
            session=session,
            session_manager=self.session_manager,
        )
    
    # Existing run() logic...
    
    # Save session if available
    if session:
        await self.session_manager.save(session)
    
    return AgentResult(output=context.final_result, context=context)

### 6.3.4 Basic example: Multi-turn conversation

**Listing 6.14** Creating an agent with session support

In [ ]:
agent = Agent(
    model=model,
    instructions="You are a helpful assistant.",
    session_manager=InMemorySessionManager()
)

### 6.3.5 Data structures for tool confirmation

**Listing 6.15** PendingToolCall class

In [ ]:
class PendingToolCall(BaseModel):
    """A tool call awaiting user confirmation."""
    
    tool_call: ToolCall
    confirmation_message: str

**Listing 6.16** ToolConfirmation class

In [ ]:
class ToolConfirmation(BaseModel):
    """User's decision on a pending tool call."""
    
    tool_call_id: str
    approved: bool
    modified_arguments: dict | None = None

**Listing 6.17** AgentResult with pending support

In [ ]:
from typing import Literal
class AgentResult(BaseModel):
    """Result of agent execution."""
    
    output: str | BaseModel | None = None
    context: ExecutionContext
    status: Literal["complete", "pending", "error"] = "complete"
    pending_tool_calls: list[PendingToolCall] = Field(default_factory=list)

### 6.3.6 Extending tools for confirmation

**Listing 6.18** BaseTool with confirmation support

In [ ]:
class BaseTool(ABC):
    # Existing attributes: name, description, parameters, execute()...
    
    # Confirmation support
    requires_confirmation: bool = False
    confirmation_message_template: str = (
        "The agent wants to execute '{name}' with arguments: {arguments}. "
        "Do you approve?"
    )
    
    def get_confirmation_message(self, arguments: dict[str, Any]) -> str:
        """Generate a confirmation message for this tool call."""
        return self.confirmation_message_template.format(
            name=self.name,
            arguments=arguments
        )

**Listing 6.19** Tools with confirmation requirements

In [ ]:
@tool(
    name="delete_file",
    description="Delete a file from the filesystem",
    requires_confirmation=True,
    confirmation_message="⚠️ The agent wants to delete '{arguments[filename]}'. "
                        "This action cannot be undone. Do you approve?"
)
def delete_file(filename: str) -> str:
    """Delete the specified file."""
    import os
    os.remove(filename)
    return f"Successfully deleted {filename}"
@tool(
    name="get_weather",
    description="Get current weather for a location"
)
def get_weather(location: str) -> str:
    """Get weather information. No confirmation needed."""
    return f"Weather in {location}: Sunny, 22°C"

### 6.3.7 Implementing pause and resume in the agent

**Listing 6.20** Modified act() method with confirmation check

In [ ]:
async def act(
    self,
    context: ExecutionContext,
    tool_calls: List[ToolCall]
) -> List[ToolResult]:
    tools_dict = {tool.name: tool for tool in self.tools}
    pending_calls = []
    for tool_call in tool_calls:
        tool = tools_dict.get(tool_call.name)
        # Check if confirmation is required
        if tool.requires_confirmation:
            pending = PendingToolCall(
                tool_call=tool_call,
                confirmation_message=tool.get_confirmation_message(
                    tool_call.arguments
                )
            )
            pending_calls.append(pending)
            continue  # Do not execute, do not create ToolResult
        # Execute tools that don't require confirmation
        # ... existing tool execution logic ...
    # Store pending calls in session state
    if pending_calls:
        context.state["pending_tool_calls"] = [
            p.model_dump() for p in pending_calls
        ]
    return results

**Listing 6.21** Modified run() method with pause and resume

In [ ]:
async def run(
    self,
    user_input: str | None = None,
    session_id: str | None = None,
    tool_confirmations: List[ToolConfirmation] | None = None,
    context: ExecutionContext | None = None,
) -> AgentResult:
    # Register confirmations for processing in _prepare_llm_request
    if tool_confirmations:
        context.state["tool_confirmations"] = [
            c.model_dump() for c in tool_confirmations
        ]
    # Main agent loop
    while not context.final_result and context.current_step < self.max_steps:
        await self.step(context)
        # Check for pending confirmations after each step
        if context.state.get("pending_tool_calls"):
            pending_calls = [
                PendingToolCall.model_validate(p)
                for p in context.state["pending_tool_calls"]
            ]
            return AgentResult(
                status="pending_confirmation",
                context=context,
                pending_tool_calls=pending_calls,
            )

**Listing 6.22** Modified _prepare_llm_request() for confirmation processing

In [ ]:
async def _prepare_llm_request(
    self,
    context: ExecutionContext
) -> LlmRequest:
    """Package context into an LlmRequest, processing confirmations first."""
    # Process pending confirmations if both are present
    if ("pending_tool_calls" in context.state and
        "tool_confirmations" in context.state):
        confirmation_results = await self._process_confirmations(context)
        # Add results as an event so they appear in contents
        if confirmation_results:
            confirmation_event = Event(
                execution_id=context.execution_id,
                author=self.name,
                content=confirmation_results,
            )
            context.add_event(confirmation_event)
        # Clear processed state
        del context.state["pending_tool_calls"]
        del context.state["tool_confirmations"]
    # Flatten events into content items
    flat_contents = []
    for event in context.events:
        flat_contents.extend(event.content)
    # ... existing LlmRequest construction logic ...

**Listing 6.23** _process_confirmations() implementation

In [ ]:
async def _process_confirmations(
    self,
    context: ExecutionContext
) -> List[ToolResult]:
    tools_dict = {tool.name: tool for tool in self.tools}
    results = []
    # Restore pending tool calls from state
    pending_map = {
        p["tool_call"]["tool_call_id"]: PendingToolCall.model_validate(p)
        for p in context.state["pending_tool_calls"]
    }
    # Build confirmation lookup by tool_call_id
    confirmation_map = {
        c["tool_call_id"]: ToolConfirmation.model_validate(c)
        for c in context.state["tool_confirmations"]
    }
    # Process ALL pending tool calls
    for tool_call_id, pending in pending_map.items():
        tool = tools_dict.get(pending.tool_call.name)
        confirmation = confirmation_map.get(tool_call_id)
        if confirmation and confirmation.approved:
            # Merge original arguments with modifications
            arguments = {
                **pending.tool_call.arguments,
                **(confirmation.modified_arguments or {})
            }
            # Execute the approved tool
            try:
                output = await tool(context, **arguments)
                results.append(ToolResult(
                    tool_call_id=tool_call_id,
                    name=pending.tool_call.name,
                    status="success",
                    content=[output],
                ))
            except Exception as e:
                results.append(ToolResult(
                    tool_call_id=tool_call_id,
                    name=pending.tool_call.name,
                    status="error",
                    content=[str(e)],
                ))
        else:
            # Rejected: either explicitly or not in confirmation list
            if confirmation:
                reason = confirmation.reason or "Tool execution was rejected by user."
            else:
                reason = "Tool execution was not approved."
            results.append(ToolResult(
                tool_call_id=tool_call_id,
                name=pending.tool_call.name,
                status="error",
                content=[reason],
            ))
    return results

### 6.3.8 Complete example: Human-in-the-loop workflow

**Listing 6.24** Setting up tools and agent for human-in-the-loop

In [ ]:
from agent import Agent, InMemorySessionManager, ToolConfirmation
from tools import tool
@tool(
    name="delete_file",
    description="Delete a file from the filesystem",
    requires_confirmation=True,
    confirmation_message="⚠️ Delete '{arguments[filename]}'? This cannot be undone."
)
def delete_file(filename: str) -> str:
    import os
    os.remove(filename)
    return f"Deleted {filename}"
@tool(name="list_files", description="List files in a directory")
def list_files(directory: str = ".") -> str:
    import os
    return "\n".join(os.listdir(directory))
agent = Agent(
    model=model,
    tools=[delete_file, list_files],
    instructions="You are a helpful file management assistant.",
    session_manager=InMemorySessionManager()
)

**Listing 6.25** Initial request triggering pending confirmation

In [ ]:
result = await agent.run(
    "Please delete the file named temp.txt",
    session_id="file_session"
)
print(f"Status: {result.status}")
# Status: pending_confirmation
print(f"Pending tools: {len(result.pending_tool_calls)}")
# Pending tools: 1
for pending in result.pending_tool_calls:
    print(f"Tool: {pending.tool_call.name}")
    print(f"Arguments: {pending.tool_call.arguments}")
    print(f"Message: {pending.confirmation_message}")
# Tool: delete_file
# Arguments: {'filename': 'temp.txt'}
# Message: ⚠️ Delete 'temp.txt'? This cannot be undone.

**Listing 6.26** Resuming execution with user approval

In [ ]:
# User approves the deletion
confirmation = ToolConfirmation(
    tool_call_id=result.pending_tool_calls[0].tool_call.tool_call_id,
    approved=True
)
result = await agent.run(
    session_id="file_session",
    tool_confirmations=[confirmation]
)
print(f"Status: {result.status}")
# Status: complete
print(f"Output: {result.output}")
# Output: I've deleted the file temp.txt as requested.

## 6.4 Long-term memory: Accumulating knowledge across sessions

### 6.4.1 The structure of long-term memory

**Listing 6.27** TaskMemory schema definition

In [ ]:
class TaskMemory(BaseModel):
    """Structured memory for GAIA problem-solving records"""
    
    task_summary: str = Field(description="What the problem asked")
    approach: str = Field(description="Methods and tools used to solve it")
    final_answer: str = Field(description="The agent's submitted answer")
    is_correct: bool = Field(description="Whether the answer was correct")
    error_analysis: str | None = Field(
        default=None, 
        description="Why the attempt failed, if it did"
    )
    def to_embedding_text(self) -> str:
        """Generate text for vector search"""
        return f"Task: {self.task_summary}"

### 6.4.2 Information extraction: Structured output

**Listing 6.28** DuplicateCheckResult schema and prompt

In [ ]:
class DuplicateCheckResult(BaseModel):
    """Result of duplicate check"""
    decision: str = Field(description="ADD (new information) or SKIP (duplicate)")
    reason: str = Field(description="Explanation for the decision")
DUPLICATE_CHECK_PROMPT = """Compare the new memory against existing memories to determine if it's a duplicate.
Existing memories:
{existing_memories}
New memory:
{new_memory}
Respond with one of:
- ADD: This is new information that should be stored
- SKIP: Similar information already exists, no need to store
Judgment criteria:
- Same problem with different approach or different result counts as new information
- Same problem with same approach and same result is a duplicate
"""

### 6.4.3 Building a vector store with ChromaDB

**Listing 6.29** ChromaDB basic usage

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
# Create client (in-memory)
client = chromadb.Client()
# Set up embedding function
embedding_fn = OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)
# Create collection
collection = client.get_or_create_collection(
    name="my_memories",
    embedding_function=embedding_fn
)
# Store documents - just pass text, embeddings are generated automatically
collection.add(
    ids=["mem_001", "mem_002"],
    documents=["The cat is sleeping on the sofa", "The dog is playing in the park"],
    metadatas=[{"category": "cat"}, {"category": "dog"}]
)
# Search - just pass query text, embeddings are generated automatically
results = collection.query(
    query_texts=["Where is the cat resting"],
    n_results=1
)
print(results["documents"])  # [['The cat is sleeping on the sofa']]
print(results["metadatas"])  # [[{'category': 'cat'}]]

### 6.4.4 Implementing TaskMemoryManager

**Listing 6.30** TaskMemory Manager initialization

In [ ]:
import uuid
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
class TaskMemoryManager:
    """Memory manager for GAIA problem-solving learning"""
    def __init__(
        self,
        llm_client: LlmClient,
        collection_name: str = "task_memories",
    ):
        self.llm_client = llm_client
        # ChromaDB setup
        self.client = chromadb.Client()
        embedding_fn = OpenAIEmbeddingFunction(
            model_name="text-embedding-3-small"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=embedding_fn
        )

**Listing 6.31** Memory extraction methods

In [ ]:
class TaskMemoryManager:
    # ... previous code ...
    async def _extract_memory(self, execution_history: str) -> TaskMemory | None:
        """Extract structured memory from execution history"""
        prompt = TASK_MEMORY_EXTRACTION_PROMPT.format(
            execution_history=execution_history
        )
        try:
            return await self.llm_client.ask(
                prompt=prompt,
                response_format=TaskMemory
            )
        except Exception as e:
            print(f"Memory extraction failed: {e}")
            return None
    def _format_execution_history(self, events: list[Event]) -> str:
        """Convert event list to text"""
        lines = []
        for event in events:
            for item in event.content:
                if isinstance(item, Message):
                    lines.append(f"[{item.role}]: {item.content}")
                elif isinstance(item, ToolCall):
                    lines.append(f"[Tool Call]: {item.name}({item.arguments})")
                elif isinstance(item, ToolResult):
                    content_preview = str(item.content[0])[:500] if item.content else ""
                    lines.append(f"[Tool Result]: {item.name} -> {content_preview}")
        return "\n".join(lines)

**Listing 6.32** Duplicate check method

In [ ]:
class TaskMemoryManager:
    # ... previous code ...
    async def _is_duplicate(
        self,
        new_memory: TaskMemory,
        existing_results: dict
    ) -> bool:
        """Determine if a new memory duplicates an existing one"""
        if not existing_results["metadatas"] or not existing_results["metadatas"][0]:
            return False
        # Convert existing memories to text
        existing_texts = []
        for meta in existing_results["metadatas"][0]:
            existing_texts.append(
                f"task_summary: {meta.get('task_summary')}, "
                f"- approach: {meta.get('approach')}, "
                f"is_correct: {meta.get('is_correct')}"
            )
        prompt = DUPLICATE_CHECK_PROMPT.format(
            existing_memories="\n".join(existing_texts),
            new_memory=f"task_summary: {new_memory.task_summary}, "
                       f"approach: {new_memory.approach}, "
                       f"is_correct: {new_memory.is_correct}"
        )
        try:
            result = await self.llm_client.ask(
                prompt=prompt,
                response_format=DuplicateCheckResult
            )
            return result.decision == "SKIP"
        except Exception:
            return False  # Proceed with storage if parsing fails

**Listing 6.33** TaskMemoryManager save() method

In [ ]:
class TaskMemoryManager:
    # ... previous code ...
    async def save(self, context: ExecutionContext) -> str | None:
        """
        Extract and save memory from execution context.
        Returns:
            memory_id if saved, None if ignored as duplicate
        """
        # 1. Convert execution history to text
        execution_history = self._format_execution_history(context.events)
        # 2. Extract structured memory using LLM
        memory = await self._extract_memory(execution_history)
        if memory is None:
            return None
        # 3. Duplicate check - search with the same text used for storage
        existing = self.collection.query(
            query_texts=[memory.to_embedding_text()],
            n_results=3
        )
        if await self._is_duplicate(memory, existing):
            return None
        # 4. Store in ChromaDB
        memory_id = str(uuid.uuid4())
        self.collection.add(
            ids=[memory_id],
            documents=[memory.to_embedding_text()],
            metadatas=[memory.model_dump()]
        )
        return memory_id

**Listing 6.34** TaskMemoryManager search() method

In [ ]:
class TaskMemoryManager:
    # ... previous code ...
    async def search(self, query: str, top_k: int = 5) -> list[TaskMemory]:
        """Search for memories related to the query."""
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )
        if not results["metadatas"] or not results["metadatas"][0]:
            return []
        return [TaskMemory(**meta) for meta in results["metadatas"][0]]

**Listing 6.35** Adding memory_manager to the Agent

In [ ]:
class Agent:
    def __init__(
        ...
        memory_manager: TaskMemoryManager | None = None,  # Added
    ):
        ...
        self.memory_manager = memory_manager  # Added

### 6.4.5 Retrieving memories

**Listing 6.36** MemoryTool for explicit retrieval

In [ ]:
class MemoryTool(BaseTool):
    """Tool for retrieving past problem-solving records"""
    def __init__(self):
        super().__init__(
            name="recall_memory",
            description=(
                "Search for past problem-solving records. "
                "Use this to check if similar problems were solved before."
            ),
        )
    async def execute(self, context: ExecutionContext, query: str) -> str:
        """Search memories and return formatted results"""
        memories = await context.memory_manager.search(query, top_k=3)
        if not memories:
            return ""
        return self._format_memories(memories)
    def _format_memories(self, memories: list[TaskMemory]) -> str:
        """Format memories for display"""
        results = []
        for i, mem in enumerate(memories, 1):
            status = "Correct" if mem.is_correct else "Incorrect"
            text = f"""[Record {i}]
- Problem: {mem.task_summary}
- Approach: {mem.approach}
- Answer: {mem.final_answer}
- Result: {status}"""
            if not mem.is_correct and mem.error_analysis:
                text += f"\n- Error analysis: {mem.error_analysis}"
            results.append(text)
        return "\n\n".join(results)

**Listing 6.37** MemoryTool with automatic injection

In [ ]:
class MemoryTool(BaseTool):
    # ... execute, _format_memories same as above ...
    def __init__(self):
        super().__init__(
            name="recall_memory",
            description="...",
            tool_definition=None,  # Automatic injection only
        )
    async def process_llm_request(
        self,
        context: ExecutionContext,
        request: LlmRequest,
    ) -> None:
        """Inject relevant memories before LLM call"""
        user_msgs = [c for c in request.contents if isinstance(c, Message) and c.role == "user"]
        if not user_msgs:
            return
        result = await self.execute(context, user_msgs[-1].content)
        if not result:
            return
        request.append_instructions(
            f"""The following are records from similar problems solved in the past:
<PAST_EXPERIENCES>
{result}
</PAST_EXPERIENCES>
Reference successful approaches and avoid approaches that led to failures."""
        )

**Listing 6.38** Modified _prepare_llm_request() for tool injection

In [ ]:
class Agent:
    # ... existing code ...
    async def _prepare_llm_request(
        self, context: ExecutionContext
    ) -> LlmRequest:
        flat_contents = []
        for event in context.events:
            flat_contents.extend(event.content)
        # Filter tools that should be exposed to the LLM
        llm_tools = [t for t in self.tools if t.tool_definition is not None]
        request = LlmRequest(
            instructions=[self.instructions] if self.instructions else [],
            contents=flat_contents,
            tools=llm_tools,
            tool_choice="auto" if llm_tools else None,
        )
        # Let tools modify the request
        for tool in self.tools:
            await tool.process_llm_request(context, request)
        return request